In [ ]:
##from google.colab import drive
#drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install tensorflow pillow numpy tqdm scikit-learn


In [ ]:
import os, math, numpy as np
from PIL import Image
from tqdm import tqdm

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model

# 1. Load model pretrained
base = MobileNetV2(weights="imagenet", include_top=False, pooling="avg", input_shape=(224,224,3))
print("✅ Model loaded. Feature dim =", base.output_shape[1])

# 2. Chuẩn bị dataset
dataset_root = "dataset"
exts = (".jpg")

paths, labels = [], []
for cls in sorted(os.listdir(dataset_root)):
    cls_dir = os.path.join(dataset_root, cls)
    if not os.path.isdir(cls_dir): continue
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(exts):
            paths.append(os.path.join(cls_dir, fname))
            labels.append(cls)

print("Tổng ảnh =", len(paths), " | Số lớp =", len(set(labels)))

# 3. Hàm load + preprocess ảnh
def load_and_preprocess(path, size=(224,224)):
    img = Image.open(path).convert("RGB")
    img = img.resize(size)
    arr = np.array(img, dtype=np.float32)
    return arr

# 4. Trích đặc trưng theo batch
batch_size = 32
features, kept_paths, kept_labels = [], [], []

for i in tqdm(range(math.ceil(len(paths)/batch_size)), desc="Extracting"):
    batch_paths = paths[i*batch_size:(i+1)*batch_size]
    imgs = [load_and_preprocess(p) for p in batch_paths]
    X = np.stack(imgs, axis=0)
    X = preprocess_input(X)
    feats = base.predict(X, verbose=0)
    features.append(feats)
    kept_paths.extend(batch_paths)
    kept_labels.extend(labels[i*batch_size:(i+1)*batch_size])

features = np.vstack(features)
print("✅ Features shape:", features.shape)

# 5. Lưu kết quả
np.save("features.npy", features)
np.save("labels.npy", np.array(kept_labels))
np.save("paths.npy", np.array(kept_paths))


✅ Model loaded. Feature dim = 1280


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DPT/dataset'

In [ ]:
import numpy as np

# Load dữ liệu
features = np.load("features.npy")
labels = np.load("labels.npy")
paths = np.load("paths.npy")

# Kiểm tra kích thước
print("✅ Features shape:", features.shape)
print("✅ Labels shape:", labels.shape)
print("✅ Paths shape:", paths.shape)

# Xem vài giá trị đầu
print("\nVí dụ 5 vector đầu tiên:")
print(features[:5])

print("\n5 nhãn đầu:")
print(labels[:5])

print("\n5 đường dẫn ảnh đầu:")
print(paths[:5])


FileNotFoundError: [Errno 2] No such file or directory: 'features.npy'